In this notebook I want to run Bayesian regression models using Bambi: 

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import arviz as az
import bambi as bmb
import pymc as pm 
from scipy.special import expit


import argparse
from os.path import join
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import zscore, spearmanr, pearsonr
import os
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns
import xarray as xr
from matplotlib.backends.backend_pdf import PdfPages


Note: If you have installed this package in editable form on Minerva, you must append the local path! This is because Minerva requires that you point your package installs away from the local directory for space reasons, but editable packages have to be installed locally.

In [ ]:
# import sys
# sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
# from LFPAnalysis import statistics_utils

## set some plotting parameters

In [ ]:
from plot_config import set_plot_style
set_plot_style()

## First, get the behavior data

This is from another notebook with a kernel and environment specifically for beahvior

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/Prolific'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/Sinai/MemBanditsEMU/Plots'
os.makedirs(save_dir_plots, exist_ok=True)


In [ ]:
# load behavioral data from Prolific subjects
learn_df = pd.read_csv(f'{load_dir}/DelayedMemoryTest/learn_df_HierarchicalRW.csv')
combined_df = pd.read_csv(f'{load_dir}/DelayedMemoryTest/full_data_df_HierarchicalRW.csv')


In [ ]:
learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)

In [ ]:
# Initial pilot data path 
load_dir = f'{base_dir}/projects/guLab/Salman/Prolific/DelayedMemoryTest'

# Load the demographics data to get 
demo_file = f'{load_dir}/demographics.csv'
demographics_df = pd.read_csv(demo_file, index_col=False, header=0)
# Some patients demos are shifted oddly
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Age'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex']
demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Sex'] = demographics_df.loc[~demographics_df[('Unnamed: 19')].isna(), 'Ethnicity simplified']
demographics_df = demographics_df[['Participant id', 'Age', 'Sex']]

demographics_df.rename(columns={'Participant id': 'participant',
                               'Age': 'age', 
                               'Sex': 'sex'}, inplace=True)

In [ ]:
demographics_df = demographics_df.loc[demographics_df.participant.isin(learn_df.participant.unique())]

In [ ]:
demographics_df[demographics_df.age!='DATA_EXPIRED'].age.astype(float).mean()


In [ ]:
demographics_df[demographics_df.age!='DATA_EXPIRED'].age.astype(float).std() 


## Figure 1A


Schematic

## Figure 1B (top) (done)

In [ ]:

f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'male', color='orange', ax=ax, errorbar='se')
sns.lineplot(data=learn_df[learn_df['good_dm_performance']], x='learn_trials', y= 'female', color='purple', ax=ax, errorbar='se')
plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
plt.ylabel('p(draw)')
plt.title(f'subject performance')
sns.despine()



In [ ]:
learn_df[learn_df['good_dm_performance']]

In [ ]:
df = learn_df[learn_df['good_dm_performance']].copy()

block_preference_model = bmb.Model(
    "gamble ~ 0 + C(block) + (1|participant)",
    data=df,
    family="bernoulli",
)
block_preference_idata = block_preference_model.fit(
    draws=2000, tune=1000, chains=4, cores=4,
    target_accept=0.95, random_seed=0, progressbar=False,
)

post = block_preference_idata.posterior
coef_var = [v for v in post.data_vars if v.startswith("C(block)")][0]
coefs = post[coef_var]
levels = list(coefs.coords[coefs.dims[-1]].values)

rows = []
for lvl in levels:
    draws = coefs.sel({coefs.dims[-1]: lvl}).values.ravel()
    hdi_lo, hdi_hi = az.hdi(draws, hdi_prob=0.94)
    p_dir = float(max((draws > 0).mean(), (draws < 0).mean()))
    rows.append({
        'block': lvl,
        'intercept_logit_mean': float(draws.mean()),
        'hdi_lo': float(hdi_lo),
        'hdi_hi': float(hdi_hi),
        'p_gamble1_mean': float(expit(draws).mean()),
        'p_direction': p_dir,
        'preferred_side': int(draws.mean() > 0),
        'credible_preference': bool((hdi_lo > 0) or (hdi_hi < 0)),
    })

block_preference_results = (
    pd.DataFrame(rows).reset_index(drop=True)
)
block_preference_results

In [ ]:
# Option 3: block-resolved Bayesian mixed logistic model
# Tests whether hit probability is above chance (0.5) in each block,
# and computes a simultaneous posterior probability for "above chance throughout blocks".

hits_block_df = combined_df[
    (combined_df.condition == 'day_1')
].dropna(subset=['hits', 'block', 'participant']).copy()

hits_block_df['hits'] = hits_block_df['hits'].astype(int)
hits_block_df['block'] = hits_block_df['block'].astype(int)

hits_block_model = bmb.Model(
    "hits ~ 0 + C(block) + (1|participant)",
    data=hits_block_df,
    family="bernoulli",
)

hits_block_idata = hits_block_model.fit(
    draws=2000,
    tune=1000,
    chains=4,
    cores=4,
    target_accept=0.95,
    random_seed=0,
    progressbar=True,
)

post = hits_block_idata.posterior
coef_var = [v for v in post.data_vars if v.startswith('C(block)')][0]
coef = post[coef_var]
block_coord = coef.coords[coef.dims[-1]].values

rows = []
block_draws_list = []
for block_label in block_coord:
    draws = coef.sel({coef.dims[-1]: block_label}).values.ravel()
    hdi_lo, hdi_hi = az.hdi(draws, hdi_prob=0.94)
    p_hit_draws = expit(draws)

    block_num = int(str(block_label).split('[')[-1].replace(']', ''))
    block_draws_list.append((block_num, p_hit_draws))

    rows.append({
        'block': block_num,
        'intercept_logit_mean': float(draws.mean()),
        'hdi_lo_logit': float(hdi_lo),
        'hdi_hi_logit': float(hdi_hi),
        'p_hit_mean': float(p_hit_draws.mean()),
        'p_hit_hdi_lo': float(expit(hdi_lo)),
        'p_hit_hdi_hi': float(expit(hdi_hi)),
        'credible_above_chance': bool(hdi_lo > 0),
    })

block_preference_results = pd.DataFrame(rows).sort_values('block').reset_index(drop=True)

# Simultaneous posterior probability that p(hit) > 0.5 in every block
block_draws_list = sorted(block_draws_list, key=lambda x: x[0])
p_matrix = np.column_stack([arr for _, arr in block_draws_list])
simul_prob_above_chance_all_blocks = float((p_matrix.min(axis=1) > 0.5).mean())

print(
    "P(p_hit(block) > 0.5 for ALL blocks | data) = "
    f"{simul_prob_above_chance_all_blocks:.3f}"
)
block_preference_results

## Figure 1B (bottom) (done)

In [ ]:

f, ax = plt.subplots(1,1, dpi=300, figsize=(3,2))
sns.lineplot(data=combined_df[combined_df.condition=='day_1'], x='learn_trials', y= 'hits', color='black', ax=ax, errorbar='se')
plt.vlines([16, 33, 48, 65], 0, 1, color = 'black')
plt.ylabel('p(subsequent hit)')
plt.title(f'subject performance')
sns.despine()

In [ ]:
# Option 1: trial-resolved Bayesian mixed logistic model
# Tests whether hit probability is above chance (0.5) at each trial,
# and computes a simultaneous posterior probability for "above chance throughout".

hits_trial_df = combined_df[
    (combined_df.condition == 'day_1')
].dropna(subset=['hits', 'learn_trials', 'participant']).copy()

hits_trial_df['hits'] = hits_trial_df['hits'].astype(int)
hits_trial_df['learn_trials'] = hits_trial_df['learn_trials'].astype(int)

hits_trial_model = bmb.Model(
    "hits ~ 0 + C(block) + (1|participant)",
    data=hits_trial_df,
    family="bernoulli",
)

hits_trial_idata = hits_trial_model.fit(
    draws=2000,
    tune=1000,
    chains=4,
    cores=4,
    random_seed=0,
    progressbar=True,
)

post = hits_trial_idata.posterior
coef_var = [v for v in post.data_vars if v.startswith('C(learn_trials)')][0]
coef = post[coef_var]
trial_coord = coef.coords[coef.dims[-1]].values

rows = []
trial_draws_list = []
for trial_label in trial_coord:
    draws = coef.sel({coef.dims[-1]: trial_label}).values.ravel()
    hdi_lo, hdi_hi = az.hdi(draws, hdi_prob=0.94)
    p_hit_draws = expit(draws)

    trial_num = int(str(trial_label).split('[')[-1].replace(']', ''))
    trial_draws_list.append((trial_num, p_hit_draws))

    rows.append({
        'learn_trial': trial_num,
        'intercept_logit_mean': float(draws.mean()),
        'hdi_lo_logit': float(hdi_lo),
        'hdi_hi_logit': float(hdi_hi),
        'p_hit_mean': float(p_hit_draws.mean()),
        'p_hit_hdi_lo': float(expit(hdi_lo)),
        'p_hit_hdi_hi': float(expit(hdi_hi)),
        'credible_above_chance': bool(hdi_lo > 0),
    })

trial_preference_results = pd.DataFrame(rows).sort_values('learn_trial').reset_index(drop=True)

# Simultaneous posterior probability that p(hit) > 0.5 at every trial
trial_draws_list = sorted(trial_draws_list, key=lambda x: x[0])
p_matrix = np.column_stack([arr for _, arr in trial_draws_list])
simul_prob_above_chance = float((p_matrix.min(axis=1) > 0.5).mean())

print(f"P(p_hit(trial) > 0.5 for ALL trials | data) = {simul_prob_above_chance:.3f}")
trial_preference_results

## Figure 1C

Schematic

## Figure 1D (done)

In [ ]:
# Load all the models and compare them:

df_comp = az.compare({'RW':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_centered'),
                      'WSLS': az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalWSLSModel'),
                     'RWUS':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Unsigned'), 
                     'RWF':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalRWModel_Fictive1'),
                     'Bayes':az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalBayesModel'),
                     'VKF': az.from_netcdf(f'{base_dir}/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/HierarchicalVKFModel')},
                     ic='waic')


In [ ]:
df_comp

In [ ]:
az.plot_compare(df_comp, figsize=[4,4], plot_ic_diff=False)

## Figure 1E (done)

In [ ]:
ax = sns.jointplot(data = learn_df[learn_df.good_dm_performance].groupby('participant').mean(numeric_only=True), 
                    x='alpha', y='beta', height=8, ratio=2, kind='kde', shade=True, cmap='Spectral_r', 
                    fill=True, cbar=True)

plt.subplots_adjust(left=0.1, right=0.8, top=0.9, bottom=0.1)
# get the current positions of the joint ax and the ax for the marginal x
pos_joint_ax = ax.ax_joint.get_position()
pos_marg_x_ax = ax.ax_marg_x.get_position()
# reposition the joint ax so it has the same width as the marginal x ax
ax.ax_joint.set_position([pos_joint_ax.x0, pos_joint_ax.y0, pos_marg_x_ax.width, pos_joint_ax.height])
# reposition the colorbar using new x positions and y positions of the joint ax
ax.fig.axes[-1].set_position([.83, pos_joint_ax.y0, .07, pos_joint_ax.height])
ax.ax_joint.set_xlabel(u"\u03B1", fontsize=14)
ax.ax_joint.set_ylabel(u"\u03B2", fontsize=14)



## Figure 1F

In [ ]:
def run_model(df, y=None, X=None, Intx=None, rand_effect=None, rand_slopes=False, within_subject_X=None,
              within_subject_Intx=None, priors=None, categorical=None,
              cores=2, chains=2, tune=1500, draws=2000, target_accept=0.95, model_fam='bernoulli', 
              output_dir='/Volumes/T7 Shield/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest', return_model=False,
              categorical_baseline=True, save_model_res = True, label=None):
    """
    Run the model: 
    
    Parameters
    ----------
    df: pandas dataframe 
        All the variables 
    y: str
        Name of the dependent variable
    X: list of strings 
        Names of the independent variables 
    Intx: list of strings
        Names of the interactions, formatted as 'a:b' 
    rand_effect: str
        Name of variable for random effects (i.e. 'subject') 
    rand_slopes: bool 
        Should we fit random slopes in addition to random intercepts? 
    priors: dict 
        Prior distribution for any independent variables (i.e. Intercept, random intercept) 
        Default to weakly informative priors (Gelman et al. 2008) 
    categorical: list of strings 
        Names of categorical variables going into model 
    
    Returns
    -------
    results: model
    
    
    y = 'memory_outcome'
    X = ['sds_score', 'zrpe', 'zpm', 'gamble_rt_log', 'recog_rt_log', 'img_cat', 'age', 'Sex', 'alpha_pos', 'alpha_neg', 'beta']
    Intx = ['sds_score:zrpe', 'sds_score:zpm']
    rand_effect = ['participant']
    categorical = ['img_cat', 'Sex']

    # Drop nan data
    model_df = model_df.dropna(subset=X)

    model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=False, 
                               priors=None, categorical=categorical, cores=4, chains=4, tune=1500, draws=2000, 
                               target_accept=0.9, model_fam='bernoulli', 
                               output_dir='/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/', 
                               return_model=True,
                               categorical_baseline=True, 
                               save_model_res = True, 
                               label=None)
    
    """

    
    # if no label provided, make your own
    if not label: 
        label = (f"{y}" + "_{}"*len(X)).format(*X)
        
    
    if categorical_baseline: # this means we set a baseline for categorical effects 
        # format random effects properly 
        rand_term = [f'(1|{x})' for x in rand_effect]
        if rand_slopes: # do we want a more complex mixed-effects model with random slopes as well as intercepts?
            if not within_subject_X:
                raise ValueError('No within subject variable specified')
            if not Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])
            elif not within_subject_Intx:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'({x}|{rand_effect[0]})' for x in within_subject_X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'({intx}|{rand_effect[0]})' for intx in within_subject_Intx])
        else:
            if not Intx: 
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 1+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)
    else: # this means all levels of categorical effect are judged absent or present 
         # format random effects properly 
        rand_term = [f'(0+1|{x})' for x in rand_effect]
        if rand_slopes: 
            formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join([f'(0 + {x}|{rand_effect[0]})' for x in X])+'+'+'+'.join(Intx)+'+'+'+'.join([f'(0 + {intx}|{rand_effect[0]})' for intx in Intx])
        else:
            if not Intx:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)
            else:
                formula = f'{y} ~ 0+'+'+'.join(rand_term)+'+'+'+'.join(X)+'+'+'+'.join(Intx)        

    print(formula)
    # construct the model 
    model = bmb.Model(formula=formula, 
                  data=df[rand_effect + [y] + X],
                 family=model_fam,
                 priors=priors,
                 categorical=categorical)
    
    # fit the model 
    results=model.fit(cores=cores, 
                          chains=chains, 
                          tune=tune, 
                          draws=draws, 
                          target_accept=target_accept)
    # init='advi+adapt_diag'

    # save out the CSV with the results 
    summary = az.summary(results, hdi_prob=0.95)
    for var in summary.index.values:
        if var.endswith("]"):  # skip individual subject parameters
            continue
        if var.startswith("1"):  # skip random effects.
            continue
        summary.loc[var, "P>0"] = np.mean(results.posterior[var].values > 0)
    summary.to_csv(join(output_dir , f"{label}_summary.csv"))

    # save out the HDI plot of all the fixed effects 
    axes = az.plot_forest(results,
                           kind='ridgeplot',
                           var_names=[f'^{x}' for x in X],
                           filter_vars="regex",
                           colors='black',
                           combined=True,
                           hdi_prob=0.95,
                           figsize=(9, 7))
    plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
    # forestplot
#         plt.xlim([-0.2, 0.1])

    plt.savefig(join(output_dir, f'{label}_HDIplot.pdf'), dpi=300)
    plt.close()

    # print out the latex table for the cell output
    results_df = summary.loc[:,
                   ['mean', 'sd', 'hdi_2.5%', 'hdi_97.5%']
                  ]
    all_tests = [] 

    for x in X: 
        temp = results_df.filter(like=f'{x}', axis=0)
        all_tests.append(temp)

    results_df = pd.concat(all_tests)

    c_string = '|c'*results_df.shape[0] + '|'
#     print(results_df.reset_index().to_latex(index=False, 
#                                             column_format=c_string).replace("\\\n", "\\ \hline\n"))  

    if save_model_res:
        # save out the model for model comparison (THIS WILL BE GB-sized, so really only needed for model comparison)
        az.to_netcdf(results, join(output_dir , f"{label}_model"))
    
    if return_model:
        return model, results 

        
def plot_res(fitted_model, X, ax=None, save_label=None): 
    """
    Plot selected posterior effects sorted from most positive mean to least.
    Effects with 95% HDI excluding 0 are colored blue.
    """

    import re
    import scipy.stats as st

    summary = az.summary(fitted_model, hdi_prob=0.95)

    rows = []
    for x in X:
        pattern = re.compile(rf'^(?!.*\|).*{re.escape(x)}.*$')
        matching = [v for v in summary.index if pattern.search(v)]
        for var in matching:
            rows.append({
                'var': var,
                'mean': float(summary.loc[var, 'mean']),
                'hdi_lo': float(summary.loc[var, 'hdi_2.5%']),
                'hdi_hi': float(summary.loc[var, 'hdi_97.5%']),
            })

    if len(rows) == 0:
        raise ValueError('No matching posterior variables found for X.')

    res_df = pd.DataFrame(rows).drop_duplicates('var')
    res_df['significant'] = (res_df['hdi_lo'] > 0) | (res_df['hdi_hi'] < 0)
    res_df = res_df.sort_values('mean', ascending=False).reset_index(drop=True)

    if ax is None:
        _, ax = plt.subplots(figsize=(9, 0.45 * len(res_df) + 2))

    y_positions = np.arange(len(res_df))

    for i, row in res_df.iterrows():
        vals = fitted_model.posterior[row['var']].values.ravel()
        kde = st.gaussian_kde(vals)
        xs = np.linspace(vals.min(), vals.max(), 400)
        dens = kde(xs)
        dens = dens / dens.max() * 0.35

        color = 'tab:blue' if row['significant'] else 'gray'
        y = y_positions[i]

        # One-sided ridge (non-symmetric density)
        ax.fill_between(xs, y, y + dens, color=color, alpha=0.45, linewidth=0)
        ax.hlines(y, row['hdi_lo'], row['hdi_hi'], color=color, linewidth=2.5)
        ax.plot(row['mean'], y, 'o', color=color, markersize=4)

    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(res_df['var'])
    ax.set_ylim(-0.25, len(res_df) - 0.25 + 0.45)
    ax.set_xlabel('Posterior effect size')
    ax.set_ylabel('Predictor')
    sns.despine(ax=ax, left=True)

    if save_label:
        plt.savefig(save_label, dpi=300, bbox_inches='tight')

    plt.show()
    return res_df

# def plot_res(fitted_model, X, ax=None, save_label=None): 
#     """
#     Code for plotting select beta weights in a publication friendly manner from saved csv
#     """
        
#     axes = az.plot_forest(fitted_model,
#                            kind='ridgeplot',
#                            var_names=[f'^(?!.*\|).*{x}.*$' for x in X],
#                            filter_vars="regex",
#                            colors='gray',
#                            combined=True,
#                            ridgeplot_overlap=2,
#                            ridgeplot_truncate=False,
#                            ridgeplot_alpha=0.4,
#                            hdi_prob=0.95,
#                            figsize=(9, 7),
#                            ax=ax)
#     # 'forestplot'
#     plt.vlines(0, plt.ylim()[0], plt.ylim()[1], color = 'black')
#     plt.show()
# #     plt.savefig(save_label, dpi=300)
# #     plt.close()

In [ ]:
model_df = combined_df[
    (combined_df.condition == 'day_1')
].dropna(subset=['hits', 'rpe', 'participant']).copy()

print(model_df.shape)
model_df = model_df.merge(demographics_df, on='participant')
model_df.rename(columns={'image_dprime':'pm'}, inplace=True)
model_df = model_df[['participant', 'hits', 'rpe', 'pm', 'sex', 'age', 'alpha', 'beta']]

In [ ]:
def _gelman_standardize(values: pd.Series) -> pd.Series:
    arr = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    mu = np.nanmean(arr)
    sd = np.nanstd(arr)
    if not np.isfinite(sd) or sd == 0:
        return pd.Series(np.nan, index=values.index)
    return pd.Series((arr - mu) / (2.0 * sd), index=values.index)

In [ ]:
# Prep a list to store results 
all_res = [] 
# Define the dependent variable and the independent variables
dependent_var = 'hits'
standardize_within_subject = ['rpe', 'pm']
standardize_between_subject = ['alpha', 'beta', 'age']
binary_standardize = ['sex']

for factor in standardize_within_subject: 
  for subj in model_df.participant.unique(): 
    subj_data = model_df[model_df.participant==subj]
    subj_data[f'z{factor}'] = _gelman_standardize(subj_data[factor])
    model_df.loc[model_df.participant==subj, f'z{factor}'] = subj_data[f'z{factor}']

for factor in standardize_between_subject: 
  model_df[f'z{factor}'] = _gelman_standardize(model_df[factor])

for factor in binary_standardize: 
  model_df[f'z{factor}'] = model_df[factor].apply(lambda x: 0.5 if x == 'Male' else -0.5)

model_df = model_df.dropna()


In [ ]:
y = 'hits'
X = ['zrpe', 'zpm', 'zalpha', 'zbeta', 'zsex', 'zage']
within_subject_X = ['zrpe', 'zpm']
Intx = []
rand_effect = ['participant']
categorical = None

label = (f"{y}" + "_{}"*len(X)).format(*X) + 'prolific_manuscript'

model, results = run_model(model_df, y=y, X=X, Intx=Intx, rand_effect=rand_effect, rand_slopes=True, within_subject_X=within_subject_X,
                           priors=None, categorical=categorical, cores=None, chains=4, tune=1500, draws=1500, 
                           target_accept=0.95, model_fam='bernoulli', 
                           output_dir='/Volumes/T7 Shield/scratch/MemoryBandit/bambi_models', 
                           return_model=True,
                           categorical_baseline=True, 
                           save_model_res = True, 
                           label=label)



In [ ]:
f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
plot_res(results, X, ax=ax, save_label=f'{save_dir_plots}/Prolific_Posteriors_Manuscript.pdf')


In [ ]:
# Load the already-run model: 
X = ['zrpe', 'zpm', 'zalpha', 'zbeta']
y= 'memory_outcome'
label = (f"{y}" + "_{}"*len(X)).format(*X) + 'exp2_day1'
results = az.from_netcdf(f'/sc/arion/projects/guLab/Salman/MemoryBanditManuscript/{label}_model')

with PdfPages(f'{save_dir_plots}/Exp2_Posteriors_Day1.pdf') as pdf:    

    f, ax = plt.subplots(1,1, dpi=300, figsize=(4,4))
    ax.set(xlabel='Weight', ylabel='Parameter')
    sns.despine()
    plot_res(results, X, ax=ax)
    
    pdf.savefig(f)

## Supplementary Figure 1:

In [ ]:
# # Run modeling and plot the graphviz
# from HierarchicalRW import HierarchicalRWModel

# n_subj = learn_df.participant.unique().shape[0]
# actions = learn_df.gamble.values.reshape(n_subj, 80).T
# rewards = learn_df.reward.values.reshape(n_subj, 80).T

# model = HierarchicalRWModel(n_subj=n_subj)
    
# m = model.make_model(actions, rewards)

# f = pm.model_to_graphviz(m)
# f.render(filename='/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/regression_model_viz',
#              format='pdf');



In [ ]:
# And the scatter plot: 

# sample from the priors (RW model) and simulate actions and rewards:

# with pm.Model() as m:
#     # Baribault parameterization: 
#     a1 = pm.Gamma(name='a1', alpha=3, beta=1)
#     a2 = pm.Gamma(name='a2', alpha=3, beta=1)
#     b1 = pm.Gamma(name='b1', alpha=5, beta=1)
#     mu_beta = pm.Normal(name='mu_beta', mu=7.5, sigma=2.5)
#     b2 = pm.Deterministic('b2', (1+b1)/mu_beta)
    
#     # centered 
#     alpha = pm.Beta(name='alpha', alpha=1+a1, beta=1+a2, shape=200)
#     beta = pm.Gamma(name='beta', alpha=1+b1, beta=b2, shape=200)   
    
# draws = pm.draw([alpha, beta], draws=3, random_seed=1)
# true_alpha = draws[0].ravel()
# true_beta = draws[1].ravel()
    
# draws = pm.draw([alpha, beta], draws=1, random_seed=1)
# true_alpha = draws[0]
# true_beta = draws[1]

# n_trials = 80
# n_subj = true_alpha.shape[0]

# mus = [[.813, .187], [.177, .823], [.93, 0.07], [0.35, 0.65], [0.73, 0.27]]
# actions = np.zeros([n_trials, n_subj], dtype="int")
# rewards = np.zeros([n_trials, n_subj], dtype="int")

# for subj in range(n_subj):
    
#     # initialise Q values for each of the two options
#     Qs = np.zeros((n_trials, 2))
#     rpe = np.zeros(n_trials)

#     # Initialize Q table
#     Q = np.array([0.5, 0.5])
#     reversal_trials = [16, 33, 48, 65]
#     block = 0
#     for t in range(n_trials):
#         if t in reversal_trials: 
#             block += 1

#         # Apply the Softmax transformation
#         exp_Q = np.exp(true_beta[subj] * Q)
#         prob_a = exp_Q / np.sum(exp_Q)

#         # Simulate choice and reward
#         a = np.random.choice([0, 1], p=prob_a)
#         r = np.random.random() < mus[block][a]

#         rpe = r - Q[a]
#         # Update Q table
#         Q[a] = Q[a] + true_alpha[subj] * rpe

#         # Store values
#         actions[t, subj] = a
#         rewards[t, subj] = r
#         Qs[t] = Q.copy()
        
# # Create a model with the fake data: 

# model = HierarchicalRWModel(n_subj=n_subj)
    
# m = model.make_model(actions, rewards)

# # Fit the model: 

# idata = model.fit(m, tune=1000, target_accept=0.8)

# # Extract the parameters: 

# sim_df = az.summary(idata)

# sim_alpha = sim_df[sim_df.index.str.startswith("alpha[")]['mean']
# sim_alpha_3 = sim_df[sim_df.index.str.startswith("alpha[")]['hdi_3%']
# sim_alpha_97 = sim_df[sim_df.index.str.startswith("alpha[")]['hdi_97%']

# sim_beta = sim_df[sim_df.index.str.startswith("beta[")]['mean']
# sim_beta_3 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_3%']
# sim_beta_97 = sim_df[sim_df.index.str.startswith("beta[")]['hdi_97%']



# Create the scatter plot
# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/parameter_recovery_alpha.pdf') as pdf:

#     f = plt.figure(figsize=(8, 8))
#     plt.errorbar(true_alpha, sim_alpha, yerr=[sim_alpha - sim_alpha_3, 
#                                          sim_alpha_97 - sim_alpha], 
#                  fmt='o', markersize=10, capsize=3, elinewidth=1, capthick=1, alpha=0.5)
#     plt.plot([0,1], [0,1], 'k--')
#     plt.xlim([0,1])
#     plt.ylim([0,1])
    
#     pdf.savefig()

# with PdfPages(f'/sc/arion/work/qasims01/MemoryBanditData/EMU/plots/parameter_recovery_beta.pdf') as pdf:

#     # Create the scatter plot
#     f = plt.figure(figsize=(8, 8))
#     plt.errorbar(true_beta, sim_beta, yerr=[sim_beta - sim_beta_3, 
#                                          sim_beta_97 - sim_beta], 
#                  fmt='o', markersize=10, capsize=3, elinewidth=1, capthick=1, alpha=0.5)
#     plt.plot([0, 15], [0, 15], 'k--')
#     plt.xlim([0, 15])
#     plt.ylim([0, 15])
    
#     pdf.savefig()